<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **2. Cellular Networking**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 2.1 Introduction

A cellular network divides a service area into cells served by base stations. Reusing spectrum in different places allows the network to serve more users. As a device moves, a handover transfers its connection between cells.

Cell planning has two main constraints: coverage, which asks whether a signal can reach a device, and capacity, which asks whether a cell can carry the users' traffic. A sparsely populated area may need several cells for coverage even when total traffic is low. A crowded area may need smaller cells to provide enough capacity.

This notebook explores these constraints with a simplified model and a hexagonal grid over a Zurich map. It illustrates trends rather than predicting actual sites or coverage. Users are uniformly distributed, propagation follows a free-space model, and every cell has the same bandwidth and assumed signal quality. Buildings, terrain, interference between cells, protocol overhead, and uplink limitations are not simulated.

<div style="display:flex;gap:12px;align-items:flex-start;width:100%;">
  <img src="../../.admin/notebooks/picture/cellular_urban_capacity.png" alt="Illustration of dense cellular coverage serving a crowded event in Zurich." style="width:calc(50% - 6px);height:auto;">
  <img src="../../.admin/notebooks/picture/cellular_rural_coverage.png" alt="Illustration of a base station providing coverage across a sparsely populated Alpine valley." style="width:calc(50% - 6px);height:auto;">
</div>

### 2.2 Theory

#### 2.2.1 Cell size from coverage

A link budget adds the gains and subtracts the losses between a transmitter and receiver. The simplified downlink model uses

$$
P_{\mathrm{RX}}=P_{\mathrm{TX}}+G_{\mathrm{TX}}+G_{\mathrm{RX}}-L_{\mathrm{path}}.
$$

Transmit and received powers are in dBm, antenna gains are in dBi, and path loss is in dB. Reception requires $P_{\mathrm{RX}}\geq P_{\mathrm{RX,min}}$, the assumed receiver sensitivity. Practical link budgets also include losses and margins that are omitted here.

For a clear free-space path, with distance $d$ in meters, frequency $f$ in hertz, and speed of light $c$ in meters per second,

$$
L_{\mathrm{path}}=20\log_{10}\left(\frac{4\pi df}{c}\right).
$$

Solving at the receiver threshold gives the coverage radius:

$$
r_{\mathrm{coverage}}=\frac{c}{4\pi f}\,10^{(P_{\mathrm{TX}}+G_{\mathrm{TX}}+G_{\mathrm{RX}}-P_{\mathrm{RX,min}})/20}.
$$

This is the free-space relationship described in [ITU-R P.525](https://www.itu.int/rec/R-REC-P.525-5-202411-I/en). With transmit power, antenna gains, and receiver sensitivity fixed, doubling the frequency halves this model's coverage radius. Increasing transmit power by about 6 dB doubles it.

Indoors, walls, reflections, and building materials introduce additional losses and variations. A higher frequency alone does not imply that coverage stops at a room boundary. The model cannot predict indoor coverage because it contains no obstacles or material properties.

#### 2.2.2 Cell size from capacity

The model estimates each cell's capacity using a Shannon-style expression:

$$
C=B\log_2(1+\gamma),
\qquad
\gamma=\frac{P_{\mathrm{signal}}}{P_{\mathrm{interference}}+P_{\mathrm{noise}}}.
$$

Here $B$ is bandwidth in hertz, $C$ is capacity in bit/s, and $\gamma$ is the signal-to-interference-and-noise ratio (SINR) as a linear power ratio. The code converts the assumed 10 dB SINR to $\gamma=10$. This is an idealized capacity estimate; actual user throughput also depends on modulation, coding, scheduling, and protocol overhead. SINR is held fixed rather than calculated from neighboring cells.

Let $\rho$ be active users per km² and $R$ the required bit/s per user. A cell of area $A$ km² has demand

$$
D_{\mathrm{cell}}=\rho A R.
$$

Requiring $D_{\mathrm{cell}}\leq C$ gives $A_{\mathrm{capacity}}=C/(\rho R)$. For a regular hexagon with center-to-vertex radius $r$ in km, $A=3\sqrt{3}r^2/2$. The capacity radius in meters is therefore

$$
r_{\mathrm{capacity}}=1000\sqrt{\frac{2C}{3\sqrt{3}\,\rho R}}.
$$

At fixed SINR, doubling bandwidth doubles the supported area; doubling user density or per-user demand halves it. Radius changes with the square root of area. Smaller cells increase capacity per unit area only under the model's assumption that each cell can reuse the same bandwidth at the same SINR.

#### 2.2.3 Combining the limits

Both constraints must be satisfied, so the model selects

$$
r_{\mathrm{cell}}=\min(r_{\mathrm{coverage}},r_{\mathrm{capacity}}).
$$

If coverage sets the smaller radius, extra bandwidth alone does not enlarge the cell in this model. If capacity sets the smaller radius, extra transmit power alone does not enlarge it. Real networks couple these effects more closely: changing power, bandwidth, or cell spacing can also change signal quality and interference.


### 2.3 Illustrative network planning for Zurich

The model calculates one radius for the entire scenario. Green cells indicate that coverage is the limiting constraint; blue cells indicate that capacity is limiting. It does not assign different traffic densities or propagation conditions to different neighborhoods.

The map shows a 5 km diameter sample area. Only cells whose centers fall inside the red boundary are counted, so the plot is an illustration rather than a complete boundary-coverage plan. The background map provides visual context and is not used for propagation calculations.

Run the helper cell, then adjust the scenario inputs and run the visualization.


In [ ]:
from pathlib import Path

helper = next(p for p in (
    Path("../../.admin/notebooks/cellular_networking.py"),
    Path(".admin/notebooks/cellular_networking.py"),
) if p.is_file())
exec(compile(helper.read_text(), str(helper), "exec"), globals())
print("Cellular planning model ready.")


#### 2.3.1 Scenario parameters

The following values are teaching assumptions. Additional model defaults in the helper are a 2.4 GHz carrier, 0 dBi antenna gain at each end, a −80 dBm receiver sensitivity, and a fixed 10 dB SINR for capacity. These values do not describe a particular Zurich operator or a licensed deployment.

##### 2.3.1.1 `tx_power`

The example uses 23 dBm, approximately 0.2 W, as the modeled transmitter power. It is not a typical macro base-station specification.

$$
P\,[\mathrm{W}]=10^{(P\,[\mathrm{dBm}]-30)/10},\qquad 23\,\mathrm{dBm}\approx0.20\,\mathrm{W}.
$$


In [ ]:
tx_power = 23              # dBm

##### 2.3.1.2 `user_density`

User density is the number of simultaneously active users per km². For illustration, the scenarios distribute 500,000 users, or 1.3 million users for a crowded event, uniformly over an assumed 75 km² service area. These are scenario inputs, not current population or attendance statistics. They imply about 6,667 and 17,333 active users/km², respectively. The map displays only a sample of that area.


In [ ]:
# number_of_Zuerchers = 1     # extreme case, only one user in the whole city
number_of_Zuerchers = 0.5e6     # illustrative baseline
#number_of_Zuerchers = 1.3e6    # illustrative crowded event

user_density = number_of_Zuerchers / 75

##### 2.3.1.3 `data_rate_per_user`

Each active user requests this data rate in bit/s. The initial value is 0.5 Mbit/s per user, assumed continuously for the capacity calculation.


In [ ]:
data_rate_per_user = .5e6   # for example, 1e6 = 1000000 = 1 Mb/s per user

##### 2.3.1.4 `bandwidth`

Bandwidth is the spectrum available to each cell, in Hertz. The example uses 100 MHz per cell and assumes reuse at the fixed SINR. Changing bandwidth affects capacity here; coverage retains its fixed receiver-sensitivity threshold.


In [ ]:
bandwidth = 100e6           # Hz 

#### 2.3.2 Visualization

Run the cell below to display both limiting radii, the selected radius, and the corresponding grid. After changing an input, rerun this cell. Compare the reported limiting constraint before interpreting a change in the map.


In [ ]:
compute_radio_planning()
network_planning_summary()
draw_hexagonal_cells(d_cell)

### 2.4 Remarks on the Model Assumptions

A transmitter power of 23 dBm is about 0.2 W. How far that signal reaches depends on frequency, antenna gains, the receiver threshold, and propagation conditions. Transmit power alone does not determine cell size.

#### 2.4.1 Path-loss model and the 23 dBm example

The model uses the free-space path-loss model [1]:

$$
L_{\mathrm{FS}}(d)=20\log_{10}\!\left(\frac{4\pi d}{\lambda}\right),
\qquad \lambda=\frac{c}{f}.
$$

Here $d$ and $\lambda$ are in meters and the loss is in dB. Received power is $P_{\mathrm{rx}}=P_{\mathrm{tx}}+G_{\mathrm{tx}}+G_{\mathrm{rx}}-L_{\mathrm{FS}}$. With 23 dBm transmit power, 0 dBi gain at each end, and a -80 dBm receiver threshold, the allowable path loss is $23-(-80)=103$ dB. At 2.4 GHz, $\lambda=0.125$ m, giving

$$
r_{\mathrm{coverage}}=\frac{0.125}{4\pi}\,10^{103/20}
\approx 1405\ \mathrm{m}.
$$

Thus the idealized coverage radius is about **1.4 km**, or **2.8 km across**. In the drawing, radius means the distance from the hexagon center to a corner. Free-space loss increases by approximately 6 dB when distance doubles.

The model selects $r_{\mathrm{cell}}=\min(r_{\mathrm{coverage}},r_{\mathrm{capacity}})$. For the notebook settings of 100 MHz per cell, 10 dB SINR, and 0.5 Mbit/s per active user, the results at 23 dBm are:

| Scenario input over 75 km² | Coverage radius | Capacity radius | Selected radius |
|---|---:|---:|---:|
| 500,000 active users | 1,405 m | 200 m | 200 m, capacity limited |
| 1.3 million active users | 1,405 m | 124 m | 124 m, capacity limited |
| One active user | 1,405 m | about 141 km | 1,405 m, coverage limited |

The very large capacity radius in the last row expresses negligible demand in the uniform-density calculation; it is not a prediction of radio reach. Calculations retain the original density even when the printed summary rounds it to a whole number. In a capacity-limited scenario, increasing transmit power alone does not enlarge the selected cell; increasing bandwidth can enlarge it until coverage becomes limiting.

Free-space propagation assumes an unobstructed path. The helper does not model building penetration, terrain blockage, shadowing, multipath fading, antenna height, or a reliability margin. Real urban coverage requires propagation assumptions suited to the environment [2, 3]. The Zurich map is a background image, not input to a propagation calculation.

Coverage and capacity are also simplified separately: coverage uses a fixed -80 dBm sensitivity, while capacity assumes a fixed 10 dB SINR everywhere. The received power at the coverage edge is therefore not checked against that capacity SINR. A more complete model would couple received power, bandwidth-dependent noise, interference, and the required data rate, and assess both uplink and downlink. Treat the radii as teaching examples rather than predictions for a Zurich deployment.

### 2.5 References and Further Reading

1. [ITU-R P.525: Calculation of free-space attenuation](https://www.itu.int/rec/R-REC-P.525-5-202411-I/en). Free download; use this to check the free-space equation and its units.
2. Andrea Goldsmith, *Wireless Communications* (2005), [Chapter 2: Path Loss and Shadowing](https://www.cambridge.org/core/books/abs/wireless-communications/path-loss-and-shadowing/A82F25D83F316AD9EE0D99DED701C9FE). Read for the distinction between ideal propagation and large-scale channel effects.
3. [3GPP TR 38.901](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3173). Optional advanced reading: compare the urban macrocell and microcell path-loss models, line-of-sight assumptions, and validity ranges with this notebook.
4. [ETH Library: searching and ETH Library @ swisscovery](https://library.ethz.ch/en/find-media/searching.html). Search for author **Andrea Goldsmith** and title **Wireless Communications** to check available editions and access options.
5. [Course background reading: Wireless Basics and Hardware](../../../BackgroundReading/Wireless%20Basics/README.md), especially the report on the Uetliberg and Felsenegg towers, for local antenna-infrastructure context.

----
### 2.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**What is the benefit of a small cell with a smaller coverage area? What is the disadvantage?**

Answer:


**Why are there so many cell towers in Zurich? Why can a telecom operator not work with just one antenna, say Üetliberg, and serve all cellphones in the area?**

Answer:


**What effect does increasing the carrier frequency have on a cell, in (a) an outdoor free space scenario, and (b) in an indoor scenario with obstacles?**

Answer:


**One active user in the service area: Run the following scenario. Which constraint determines cell size? Explain why covering the whole area can still require several cells. How do extra bandwidth and extra transmit power affect this model?**

Answer:


In [ ]:
tx_power = 23                                # dBm
number_of_Zuerchers = 1                   # extreme case, only one user in the whole city
user_density = number_of_Zuerchers / 75
data_rate_per_user = .5e6
bandwidth = 100e6 

compute_radio_planning()
network_planning_summary()
draw_hexagonal_cells(d_cell)

**Run the following illustrative Street Parade scenario. Why does the model produce smaller cells? How do extra bandwidth and extra transmit power affect the limiting constraint?**

Answer:


In [ ]:
tx_power = 23                                # dBm
number_of_Zuerchers = 1.3e6                  # illustrative crowded event
user_density = number_of_Zuerchers / 75
data_rate_per_user = .5e6
bandwidth = 100e6 

compute_radio_planning()
network_planning_summary()
draw_hexagonal_cells(d_cell)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

Answer:
